# 1.4-1.6 Window cutting & Missing data imputation & Patient-level split

## 1.4 Windows cutting (30, 60, 90, 2h, 3h, 4h, 6h) with 50% overlap


Cut into superwindows

In [ ]:
import pickle
import numpy as np
import pandas as pd
from typing import List, Optional


def superwindow_crop_pkl_timebased(
    input_pkl_path: str,
    output_pkl_path: str,
    id_col: str = "ids__uid",
    timestamp_col: str = "timestamp",

    # Frame / hop definition
    frame_len_minutes: int = 10,
    frame_hop_minutes: int = 5,

    # Superwindow definition
    superwindow_minutes: int = 180,
    super_stride_minutes: int = 60,

    # QC: per-feature minimum valid frames within the superwindow
    min_valid_frames_per_feature: int = 28,

    # Timestamp alignment
    align_timestamps: bool = True,
    align_method: str = "round",  # "round" or "floor"

    # Target aggregation
    target_col: str = "target__pma_w",

    # Feature selection
    feature_prefix: str = "feats__",

    # ✅ Static columns (patient-level constants to be filled across the grid)
    static_cols: Optional[List[str]] = None,

    # Whether QC should ignore static columns (recommended)
    qc_ignore_static: bool = True,

    # Print warnings for patients with missing/multiple static values
    verbose_static_warnings: bool = True,
):
    """
    Time-based superwindow cropping per patient using timestamps.

    Features:
    - Only columns starting with `feature_prefix` (e.g. 'feats__') are used as features.
    - timestamp_col is used ONLY for slicing, not as a feature.

    QC rule:
    - For a superwindow to be kept, EVERY (time-varying) feature must have at least
      `min_valid_frames_per_feature` non-NaN values among the frames in the grid.
      (By default, static columns are excluded from QC.)

    Static fill (Architecture #2):
    - Columns listed in `static_cols` (e.g., GA/sex/birthweight) are treated as
      patient-level constants and are filled across the entire time grid for each window.
      This prevents them from becoming NaN due to missing timestamps after reindexing.

    Target:
    - Compute mean of `target_col` within the superwindow grid (ignore NaNs),
      saved in meta and also in y_by_patient.
    """
    if static_cols is None:
        static_cols = ["feats__ga_w", "feats__sex", "feats__bw"]

    # -------------------------
    # Load data
    # -------------------------
    with open(input_pkl_path, "rb") as f:
        df_all = pickle.load(f)

    if not isinstance(df_all, pd.DataFrame):
        raise ValueError("Expected input pkl to contain a pandas DataFrame.")

    df_all = df_all.copy()
    df_all[timestamp_col] = pd.to_datetime(df_all[timestamp_col])

    if target_col not in df_all.columns:
        raise ValueError(f"target_col '{target_col}' not found in dataframe columns.")

    # -------------------------
    # Select features by prefix
    # -------------------------
    feature_cols = [
        c for c in df_all.columns
        if c.startswith(feature_prefix)
        and c != "feats__pna_days"
        and c != "feats__weight"
    ]
    
    print("Number of selected features:", len(feature_cols))
    print("Excluded weight feature:", "feats__weight" in df_all.columns and "feats__weight" not in feature_cols)
    if len(feature_cols) == 0:
        raise ValueError(f"No feature columns found with prefix '{feature_prefix}'.")

    # Ensure static cols are included as features if present in df_all
    # (If a static col is not in feature_cols but exists in df_all, we append it.)
    for c in static_cols:
        if c in df_all.columns and c not in feature_cols:
            feature_cols.append(c)

    # Sort by patient then timestamp
    df_all = df_all.sort_values([id_col, timestamp_col], kind="stable").reset_index(drop=True)

    # -------------------------
    # Convert minutes -> slot counts
    # -------------------------
    frames_per_super = int((superwindow_minutes - frame_len_minutes) / frame_hop_minutes) + 1  # e.g. 35
    if frames_per_super <= 0:
        raise ValueError("superwindow_minutes too small compared to frame_len_minutes.")
    if min_valid_frames_per_feature > frames_per_super:
        raise ValueError("min_valid_frames_per_feature cannot exceed frames_per_super.")

    hop = pd.Timedelta(minutes=frame_hop_minutes)
    super_len = pd.Timedelta(minutes=superwindow_minutes)
    super_stride = pd.Timedelta(minutes=super_stride_minutes)

    # -------------------------
    # Helper: align timestamps to hop grid
    # -------------------------
    def align_to_hop_grid(ts: pd.Series) -> pd.Series:
        mins = (ts.astype("int64") // (60 * 10**9)).astype(np.int64)
        h = frame_hop_minutes
        if align_method == "round":
            aligned = ((mins + h // 2) // h) * h
        elif align_method == "floor":
            aligned = (mins // h) * h
        else:
            raise ValueError("align_method must be 'round' or 'floor'.")
        return pd.to_datetime(aligned, unit="m")

    # -------------------------
    # Outputs + stats
    # -------------------------
    X_by_patient = {}
    y_by_patient = {}
    meta_by_patient = {}
    stats_rows = []
    total_before = 0
    total_after = 0

    # -------------------------
    # Process each patient independently
    # -------------------------
    for pid, df_p in df_all.groupby(id_col, sort=False):
        df_p = df_p.copy()

        # Align timestamps
        if align_timestamps:
            df_p["_frame_time"] = align_to_hop_grid(df_p[timestamp_col])
        else:
            df_p["_frame_time"] = df_p[timestamp_col]

        # If multiple rows map to the same frame_time, keep the first
        df_p = df_p.drop_duplicates(subset=["_frame_time"], keep="first")
        df_p = df_p.set_index("_frame_time").sort_index()

        # -------------------------
        # Extract patient-level static values ONCE
        # -------------------------
        static_vals = {}
        for c in static_cols:
            if c in df_p.columns:
                non_na = df_p[c].dropna()
                if len(non_na) == 0:
                    static_vals[c] = np.nan
                else:
                    # check uniqueness (optional warning)
                    uniq = non_na.unique()
                    if verbose_static_warnings and len(uniq) > 1:
                        print(f"[PID {pid}] WARNING: static col '{c}' has multiple values: {uniq[:10]} (n={len(uniq)})")
                    static_vals[c] = float(uniq[0]) if np.issubdtype(non_na.dtype, np.number) else uniq[0]
            else:
                static_vals[c] = np.nan

        if verbose_static_warnings:
            missing_static = [c for c, v in static_vals.items() if (isinstance(v, float) and not np.isfinite(v))]
            if len(missing_static) > 0:
                # not fatal; just warn
                print(f"[PID {pid}] WARNING: missing static values for: {missing_static}")

        # Features + target
        feat_df = df_p[feature_cols]      # features only
        target_s = df_p[target_col]       # target series for aggregation

        if feat_df.shape[0] == 0:
            X_by_patient[pid] = np.empty((0, frames_per_super, len(feature_cols)), dtype=float)
            y_by_patient[pid] = np.empty((0,), dtype=float)
            meta_by_patient[pid] = pd.DataFrame(
                columns=["super_id", "start_time", "end_time", f"{target_col}_mean"] + static_cols
            )
            stats_rows.append({"patient_id": pid, "n_super_before": 0, "n_super_after": 0, "retention_ratio": np.nan})
            continue

        t_min = feat_df.index.min()
        t_max = feat_df.index.max()

        # Candidate superwindow starts
        starts = []
        cur = t_min
        while cur <= t_max:
            starts.append(cur)
            cur = cur + super_stride

        n_super_before = len(starts)
        total_before += n_super_before

        blocks = []
        y_list = []
        meta_rows = []

        # Define QC columns
        if qc_ignore_static:
            qc_cols = [c for c in feature_cols if c not in static_cols]
            if len(qc_cols) == 0:
                # fallback: if everything is static (unlikely), QC on all
                qc_cols = list(feature_cols)
        else:
            qc_cols = list(feature_cols)

        for s_time in starts:
            grid = pd.date_range(start=s_time, periods=frames_per_super, freq=hop)

            # Reindex to fixed grid (missing slots -> NaNs)
            block_df = feat_df.reindex(grid)  # (T, F)

            # ✅ Fill static columns as constants across the grid
            for c in static_cols:
                if c in block_df.columns:
                    block_df[c] = static_vals[c]

            # QC: per-feature valid count (on qc_cols)
            per_feature_valid_counts = block_df[qc_cols].notna().sum(axis=0)
            if (per_feature_valid_counts < min_valid_frames_per_feature).any():
                continue

            # Target mean over the same grid (ignore NaNs)
            target_block = target_s.reindex(grid).to_numpy(dtype=float)
            m = np.nanmean(target_block)
            target_mean = float(m) if np.isfinite(m) else np.nan

            blocks.append(block_df.to_numpy(dtype=float))
            y_list.append(target_mean)

            meta_row = {
                "super_id": len(blocks) - 1,
                "start_time": s_time,
                "end_time": s_time + super_len,
                f"{target_col}_mean": target_mean,
            }
            # store static in meta for debugging / future use
            for c in static_cols:
                meta_row[c] = static_vals.get(c, np.nan)
            meta_rows.append(meta_row)

        n_super_after = len(blocks)
        total_after += n_super_after

        X_super = (
            np.stack(blocks, axis=0)
            if n_super_after > 0
            else np.empty((0, frames_per_super, len(feature_cols)), dtype=float)
        )
        y_super = np.array(y_list, dtype=float) if n_super_after > 0 else np.empty((0,), dtype=float)

        X_by_patient[pid] = X_super
        y_by_patient[pid] = y_super
        meta_by_patient[pid] = pd.DataFrame(meta_rows)

        stats_rows.append({
            "patient_id": pid,
            "n_super_before": n_super_before,
            "n_super_after": n_super_after,
            "retention_ratio": (n_super_after / n_super_before) if n_super_before > 0 else np.nan,
        })

    stats_df = pd.DataFrame(stats_rows)

    out = {
        "params": {
            "id_col": id_col,
            "timestamp_col": timestamp_col,
            "feature_prefix": feature_prefix,
            "target_col": target_col,
            "frame_len_minutes": frame_len_minutes,
            "frame_hop_minutes": frame_hop_minutes,
            "superwindow_minutes": superwindow_minutes,
            "super_stride_minutes": super_stride_minutes,
            "frames_per_super": frames_per_super,
            "min_valid_frames_per_feature": min_valid_frames_per_feature,
            "align_timestamps": align_timestamps,
            "align_method": align_method,
            "static_cols": list(static_cols),
            "qc_ignore_static": qc_ignore_static,
        },
        "feature_cols": feature_cols,
        "X_by_patient": X_by_patient,
        "y_by_patient": y_by_patient,
        "meta_by_patient": meta_by_patient,
        "stats_per_patient": stats_df,
        "stats_global": {
            "total_super_before": int(total_before),
            "total_super_after": int(total_after),
            "global_retention_ratio": (total_after / total_before) if total_before > 0 else np.nan,
        }
    }

    with open(output_pkl_path, "wb") as f:
        pickle.dump(out, f, protocol=pickle.HIGHEST_PROTOCOL)

    return out

In [ ]:
import os
import math
import gc
import pandas as pd


# ============================================================
# Batch run window length comparison with fixed 50% overlap
# ============================================================

input_pkl_path = "/mnt/home/qinqiu/thesis/data/data_patients_filtered__no_los_nec__ga_le_35__no_intubation.pkl"

output_dir = "/mnt/home/qinqiu/thesis/data/superwindow_windowlength_50pct_overlap_no_los_nec_ga_le_35_no_intubation"
os.makedirs(output_dir, exist_ok=True)


def calculate_min_valid_frames_loose(
    superwindow_minutes: int,
    frame_len_minutes: int = 10,
    frame_hop_minutes: int = 5,
    valid_ratio: float = 0.8,
):
    """
    Calculate the minimum number of valid frames required per feature.

    Rule:
    - Use 80% of frames_per_super.
    - If the result is not an integer, use floor instead of ceil.
      This makes the QC slightly more relaxed.
    """
    frames_per_super = int((superwindow_minutes - frame_len_minutes) / frame_hop_minutes) + 1

    if frames_per_super <= 0:
        raise ValueError("superwindow_minutes is too small compared to frame_len_minutes.")

    min_valid_frames = math.floor(frames_per_super * valid_ratio + 1e-9)

    # Ensure at least one valid frame is required
    min_valid_frames = max(1, min_valid_frames)

    return frames_per_super, min_valid_frames


# ============================================================
# Window length comparison
# Fixed overlap = 50%
# ============================================================
settings = [
    ("30min_stride15min",     30, 15),
    ("60min_stride30min",     60, 30),
    ("90min_stride45min",     90, 45),
    ("120min_stride60min",   120, 60),
    ("3h_stride90min",       180, 90),
    ("4h_stride2h",          240, 120),
    ("6h_stride3h",          360, 180),
]


summary_rows = []

for label, superwindow_minutes, super_stride_minutes in settings:
    frames_per_super, min_valid_frames = calculate_min_valid_frames_loose(
        superwindow_minutes=superwindow_minutes,
        frame_len_minutes=10,
        frame_hop_minutes=5,
        valid_ratio=0.8,
    )

    output_pkl_path = os.path.join(
        output_dir,
        f"data_patients_superwindow_{label}_timebased_featsQC_loose80.pkl"
    )

    print("=" * 80)
    print(f"Running setting: {label}")
    print(f"superwindow_minutes = {superwindow_minutes}")
    print(f"super_stride_minutes = {super_stride_minutes}")
    print(f"frames_per_super = {frames_per_super}")
    print(f"min_valid_frames_per_feature = {min_valid_frames}")
    print(f"valid_ratio_actual = {min_valid_frames / frames_per_super:.3f}")
    print(f"output = {output_pkl_path}")

    out = superwindow_crop_pkl_timebased(
        input_pkl_path=input_pkl_path,
        output_pkl_path=output_pkl_path,
        id_col="ids__uid",
        timestamp_col="timestamp",

        frame_len_minutes=10,
        frame_hop_minutes=5,

        superwindow_minutes=superwindow_minutes,
        super_stride_minutes=super_stride_minutes,

        min_valid_frames_per_feature=min_valid_frames,

        target_col="target__pma_w",
        feature_prefix="feats__",

        static_cols=["feats__ga_w", "feats__sex", "feats__bw"],
        qc_ignore_static=True,
        verbose_static_warnings=True,
    )

    global_stats = out["stats_global"]

    summary_rows.append({
        "label": label,
        "superwindow_minutes": superwindow_minutes,
        "super_stride_minutes": super_stride_minutes,
        "overlap_ratio": 1 - (super_stride_minutes / superwindow_minutes),
        "frames_per_super": frames_per_super,
        "min_valid_frames_per_feature": min_valid_frames,
        "valid_ratio_actual": min_valid_frames / frames_per_super,
        "total_super_before": global_stats["total_super_before"],
        "total_super_after": global_stats["total_super_after"],
        "global_retention_ratio": global_stats["global_retention_ratio"],
        "output_pkl_path": output_pkl_path,
    })

    print("Global stats:")
    print(global_stats)
    print()

    # Free memory after each setting
    del out
    gc.collect()


summary_df = pd.DataFrame(summary_rows)

summary_csv_path = os.path.join(
    output_dir,
    "superwindow_windowlength_50pct_overlap_summary_loose80.csv"
)

summary_df.to_csv(summary_csv_path, index=False)

print("=" * 80)
print("Window length batch cropping finished.")
print(f"Summary saved to: {summary_csv_path}")
print(summary_df)

## 1.5 Missing data imputation

data imputation: use time-based interpolation, use missingness mask, convert to 2D data

In [ ]:
import os
import gc
import pickle
import numpy as np
import pandas as pd
from typing import Dict, Any


# =========================================================
# Paths
# =========================================================
INPUT_DIR = "/mnt/home/qinqiu/thesis/data/superwindow_windowlength_50pct_overlap_no_los_nec_ga_le_35_no_intubation"

OUTPUT_DIR = "/mnt/home/qinqiu/thesis/data/superwindow_windowlength_50pct_overlap_no_los_nec_ga_le_35_no_intubation_imputed_2ch_CTHW"

os.makedirs(OUTPUT_DIR, exist_ok=True)

DROP_LENGTH_MISMATCH = True


# =========================================================
# Batch settings
# Window length comparison with fixed 50% overlap
# =========================================================
SETTINGS = [
    "30min_stride15min",
    "60min_stride30min",
    "90min_stride45min",
    "120min_stride60min",
    "3h_stride90min",
    "4h_stride2h",
    "6h_stride3h",
]

# =========================================================
# Helper functions
# =========================================================
def safe_n_windows(X):
    if X is None:
        return 0
    try:
        return int(len(X))
    except Exception:
        return 0


def safe_n_labels(y):
    if y is None:
        return 0
    try:
        return int(len(y))
    except Exception:
        return 0


def get_shape(x):
    if x is None:
        return None
    try:
        return tuple(np.shape(x))
    except Exception:
        return None


def get_invalid_reason(X, y):
    """
    Return None if patient is valid.
    Otherwise return the reason for removal.
    """
    if X is None:
        return "X is None"

    if y is None:
        return "y is None"

    if safe_n_windows(X) == 0:
        return "X has zero windows"

    if safe_n_labels(y) == 0:
        return "y has zero labels"

    if not hasattr(X, "ndim"):
        return "X has no ndim attribute"

    if X.ndim != 3:
        return f"X ndim is not 3: X.ndim={X.ndim}, X.shape={X.shape}"

    if DROP_LENGTH_MISMATCH and len(X) != len(y):
        return f"X/y length mismatch: len(X)={len(X)}, len(y)={len(y)}"

    return None


def _impute_linear_time_with_mask(x_tf: np.ndarray):
    """
    x_tf: (T, F) float array with NaNs.

    Returns:
      x_imputed_tf: (T, F) float32, no NaNs
      mask_tf:      (T, F) float32, 1 where original was NaN, else 0

    Imputation:
      Per-feature linear interpolation over time.
      Edge NaNs are filled with nearest valid value by np.interp.
      If one feature is entirely NaN, it is filled with 0.
    """
    if x_tf.ndim != 2:
        raise ValueError(f"Expected (T,F), got {x_tf.shape}")

    T, F = x_tf.shape

    mask_tf = np.isnan(x_tf).astype(np.float32)
    x_imputed = x_tf.astype(np.float32, copy=True)

    t_idx = np.arange(T, dtype=np.float32)

    for f in range(F):
        col = x_imputed[:, f]
        nan = np.isnan(col)

        if not nan.any():
            continue

        valid = ~nan

        if valid.sum() == 0:
            col[:] = 0.0
            x_imputed[:, f] = col
            continue

        col[nan] = np.interp(
            t_idx[nan],
            t_idx[valid],
            col[valid]
        ).astype(np.float32)

        if np.isnan(col).any():
            m = np.nanmean(col)
            col[np.isnan(col)] = 0.0 if not np.isfinite(m) else np.float32(m)

        x_imputed[:, f] = col

    return x_imputed, mask_tf


# =========================================================
# Main conversion function for one PKL
# =========================================================
def convert_pkl_to_imputed_2ch_cthw_cleaned(
    in_pkl: str,
    out_pkl: str,
    setting_label: str = None,
) -> Dict[str, Any]:

    with open(in_pkl, "rb") as f:
        out = pickle.load(f)

    X_by_patient = out["X_by_patient"]
    y_by_patient = out["y_by_patient"]
    feature_cols = out.get("feature_cols", None)

    params = out.get("params", {})
    frames_per_super = params.get("frames_per_super", None)
    superwindow_minutes = params.get("superwindow_minutes", None)
    super_stride_minutes = params.get("super_stride_minutes", None)

    # -----------------------------------------------------
    # Original patient counts
    # -----------------------------------------------------
    original_X_patient_count = len(X_by_patient)
    original_y_patient_count = len(y_by_patient)

    x_pids = set(X_by_patient.keys())
    y_pids = set(y_by_patient.keys())

    candidate_pids = sorted(list(x_pids.intersection(y_pids)), key=str)
    x_only_pids = sorted(list(x_pids - y_pids), key=str)
    y_only_pids = sorted(list(y_pids - x_pids), key=str)

    original_candidate_patient_count = len(candidate_pids)

    # -----------------------------------------------------
    # Original window / label counts before filtering
    # -----------------------------------------------------
    original_candidate_X_windows = 0
    original_candidate_y_labels = 0

    for pid in candidate_pids:
        X = X_by_patient.get(pid, None)
        y = y_by_patient.get(pid, None)

        original_candidate_X_windows += safe_n_windows(X)
        original_candidate_y_labels += safe_n_labels(y)

    # -----------------------------------------------------
    # Containers
    # -----------------------------------------------------
    X2_by_patient = {}
    X_by_patient_clean = {}
    y_by_patient_clean = {}

    removed_rows = []

    stats = {
        "total_windows_imputed": 0,
        "total_nan_before": 0,
        "total_nan_after": 0,
    }

    # -----------------------------------------------------
    # Filter + impute
    # -----------------------------------------------------
    for pid in candidate_pids:
        X = X_by_patient.get(pid, None)
        y = y_by_patient.get(pid, None)

        reason = get_invalid_reason(X, y)

        if reason is not None:
            removed_rows.append({
                "pid": pid,
                "reason": reason,
                "X_shape": get_shape(X),
                "y_shape": get_shape(y),
                "n_X_windows": safe_n_windows(X),
                "n_y_labels": safe_n_labels(y),
            })
            continue

        N, T, F = X.shape

        X2 = np.empty((N, 2, T, F), dtype=np.float32)

        nan_before = int(np.isnan(X).sum())
        stats["total_windows_imputed"] += int(N)
        stats["total_nan_before"] += nan_before

        for i in range(N):
            x_tf = X[i]  # (T, F)

            x_imp_tf, mask_tf = _impute_linear_time_with_mask(x_tf)

            # Channel 0 = imputed values
            # Channel 1 = missingness mask
            X2[i, 0, :, :] = x_imp_tf
            X2[i, 1, :, :] = mask_tf

        nan_after = int(np.isnan(X2[:, 0, :, :]).sum())
        stats["total_nan_after"] += nan_after

        X2_by_patient[pid] = X2
        X_by_patient_clean[pid] = X
        y_by_patient_clean[pid] = y

    # -----------------------------------------------------
    # Removed and final counts
    # -----------------------------------------------------
    removed_columns = [
        "pid",
        "reason",
        "X_shape",
        "y_shape",
        "n_X_windows",
        "n_y_labels",
    ]

    removed_df = pd.DataFrame(removed_rows, columns=removed_columns)

    if len(removed_df) > 0:
        removed_df = removed_df.sort_values("pid").reset_index(drop=True)
        removed_patient_count = int(len(removed_df))
        removed_X_windows = int(removed_df["n_X_windows"].sum())
        removed_y_labels = int(removed_df["n_y_labels"].sum())
    else:
        removed_patient_count = 0
        removed_X_windows = 0
        removed_y_labels = 0

    final_patient_count = int(len(X2_by_patient))
    final_X_windows = int(sum(len(X) for X in X_by_patient_clean.values()))
    final_y_labels = int(sum(len(y) for y in y_by_patient_clean.values()))

    # -----------------------------------------------------
    # Save output PKL
    # -----------------------------------------------------
    out2 = dict(out)

    out2["X_by_patient"] = X_by_patient_clean
    out2["y_by_patient"] = y_by_patient_clean
    out2["X2_by_patient"] = X2_by_patient

    out2["feature_cols"] = feature_cols

    out2["imputation"] = {
        "setting_label": setting_label,
        "method": "per-feature linear interpolation over time + missingness mask channel",
        "output_format": "N,C,H,W with C=2; channel 0 = imputed values, channel 1 = missingness mask",
        "invalid_patients_removed": True,
        "drop_length_mismatch": DROP_LENGTH_MISMATCH,
    }

    out2["imputation_stats"] = stats

    out2["patient_window_filtering_summary"] = {
        "setting_label": setting_label,

        "superwindow_minutes": superwindow_minutes,
        "super_stride_minutes": super_stride_minutes,
        "frames_per_super": frames_per_super,

        "original_X_by_patient_count": int(original_X_patient_count),
        "original_y_by_patient_count": int(original_y_patient_count),
        "original_candidate_patient_count_intersection_X_y": int(original_candidate_patient_count),
        "x_only_patient_count": int(len(x_only_pids)),
        "y_only_patient_count": int(len(y_only_pids)),

        "original_candidate_X_windows": int(original_candidate_X_windows),
        "original_candidate_y_labels": int(original_candidate_y_labels),

        "removed_patient_count": int(removed_patient_count),
        "removed_X_windows": int(removed_X_windows),
        "removed_y_labels": int(removed_y_labels),

        "final_patient_count": int(final_patient_count),
        "final_X_windows": int(final_X_windows),
        "final_y_labels": int(final_y_labels),
    }

    out2["removed_patients"] = removed_rows
    out2["x_only_pids"] = x_only_pids
    out2["y_only_pids"] = y_only_pids

    with open(out_pkl, "wb") as f:
        pickle.dump(out2, f, protocol=pickle.HIGHEST_PROTOCOL)

    # -----------------------------------------------------
    # Save CSV logs
    # -----------------------------------------------------
    out_dir = os.path.dirname(out_pkl)
    out_base = os.path.splitext(os.path.basename(out_pkl))[0]

    removed_csv = os.path.join(out_dir, f"{out_base}_removed_patients.csv")
    valid_csv = os.path.join(out_dir, f"{out_base}_valid_patients.csv")
    summary_txt = os.path.join(out_dir, f"{out_base}_filtering_summary.txt")

    removed_df.to_csv(removed_csv, index=False)

    valid_rows = []
    for pid in sorted(X2_by_patient.keys(), key=str):
        X2 = X2_by_patient[pid]
        y = y_by_patient_clean[pid]

        valid_rows.append({
            "pid": pid,
            "X2_shape": tuple(X2.shape),
            "n_windows": len(X2),
            "n_y_labels": len(y),
            "target_mean": float(np.nanmean(y)) if len(y) > 0 else np.nan,
            "target_median": float(np.nanmedian(y)) if len(y) > 0 else np.nan,
        })

    valid_columns = [
        "pid",
        "X2_shape",
        "n_windows",
        "n_y_labels",
        "target_mean",
        "target_median",
    ]

    valid_df = pd.DataFrame(valid_rows, columns=valid_columns)

    if len(valid_df) > 0:
        valid_df = valid_df.sort_values("pid").reset_index(drop=True)

    valid_df.to_csv(valid_csv, index=False)

    # -----------------------------------------------------
    # Print summary
    # -----------------------------------------------------
    print("\n" + "=" * 90)
    print(f"Patient and window filtering summary: {setting_label}")
    print("=" * 90)

    print("\nWindow setting:")
    print(f"superwindow_minutes:                               {superwindow_minutes}")
    print(f"super_stride_minutes:                              {super_stride_minutes}")
    print(f"frames_per_super:                                  {frames_per_super}")

    print("\nPatient counts:")
    print(f"Original X_by_patient count:                         {original_X_patient_count}")
    print(f"Original y_by_patient count:                         {original_y_patient_count}")
    print(f"Original candidate patients, intersection of X and y: {original_candidate_patient_count}")
    print(f"Patients only in X_by_patient:                       {len(x_only_pids)}")
    print(f"Patients only in y_by_patient:                       {len(y_only_pids)}")
    print(f"Removed patients:                                    {removed_patient_count}")
    print(f"Final valid patients:                                {final_patient_count}")

    print("\nWindow / label counts:")
    print(f"Original candidate X windows:                        {original_candidate_X_windows}")
    print(f"Original candidate y labels:                         {original_candidate_y_labels}")
    print(f"Removed X windows:                                   {removed_X_windows}")
    print(f"Removed y labels:                                    {removed_y_labels}")
    print(f"Final X windows:                                     {final_X_windows}")
    print(f"Final y labels:                                      {final_y_labels}")

    print("\nImputation stats:")
    print(f"Total windows imputed:                               {stats['total_windows_imputed']}")
    print(f"Total NaNs before imputation:                        {stats['total_nan_before']}")
    print(f"Total NaNs after imputation:                         {stats['total_nan_after']}")

    print("\nRemoved patients:")
    if len(removed_df) > 0:
        print(removed_df.to_string(index=False))
    else:
        print("None")

    print("\nSaved files:")
    print(f"Cleaned/imputed PKL:                                 {out_pkl}")
    print(f"Removed patients CSV:                                {removed_csv}")
    print(f"Valid patients CSV:                                  {valid_csv}")

    # -----------------------------------------------------
    # Save TXT summary
    # -----------------------------------------------------
    with open(summary_txt, "w") as f:
        f.write(f"Patient and window filtering summary: {setting_label}\n")
        f.write("=" * 90 + "\n\n")

        f.write("Window setting:\n")
        f.write(f"superwindow_minutes: {superwindow_minutes}\n")
        f.write(f"super_stride_minutes: {super_stride_minutes}\n")
        f.write(f"frames_per_super: {frames_per_super}\n\n")

        f.write("Patient counts:\n")
        f.write(f"Original X_by_patient count: {original_X_patient_count}\n")
        f.write(f"Original y_by_patient count: {original_y_patient_count}\n")
        f.write(f"Original candidate patients, intersection of X and y: {original_candidate_patient_count}\n")
        f.write(f"Patients only in X_by_patient: {len(x_only_pids)}\n")
        f.write(f"Patients only in y_by_patient: {len(y_only_pids)}\n")
        f.write(f"Removed patients: {removed_patient_count}\n")
        f.write(f"Final valid patients: {final_patient_count}\n\n")

        f.write("Window / label counts:\n")
        f.write(f"Original candidate X windows: {original_candidate_X_windows}\n")
        f.write(f"Original candidate y labels: {original_candidate_y_labels}\n")
        f.write(f"Removed X windows: {removed_X_windows}\n")
        f.write(f"Removed y labels: {removed_y_labels}\n")
        f.write(f"Final X windows: {final_X_windows}\n")
        f.write(f"Final y labels: {final_y_labels}\n\n")

        f.write("Imputation stats:\n")
        f.write(f"Total windows imputed: {stats['total_windows_imputed']}\n")
        f.write(f"Total NaNs before imputation: {stats['total_nan_before']}\n")
        f.write(f"Total NaNs after imputation: {stats['total_nan_after']}\n\n")

        f.write("Removed patients:\n")
        if len(removed_df) > 0:
            f.write(removed_df.to_string(index=False))
        else:
            f.write("None\n")

    print(f"Filtering summary TXT:                               {summary_txt}")

    return out2


# =========================================================
# Batch run
# =========================================================
if __name__ == "__main__":

    batch_summary_rows = []

    for label in SETTINGS:
        in_pkl = os.path.join(
            INPUT_DIR,
            f"data_patients_superwindow_{label}_timebased_featsQC_loose80.pkl"
        )

        out_pkl = os.path.join(
            OUTPUT_DIR,
            f"data_patients_superwindow_{label}_timebased_featsQC_loose80_imputed_2ch_CTHW.pkl"
        )

        print("\n" + "#" * 100)
        print(f"Running imputation for setting: {label}")
        print("#" * 100)
        print(f"Input:  {in_pkl}")
        print(f"Output: {out_pkl}")

        if not os.path.exists(in_pkl):
            print(f"WARNING: Input file not found. Skipping: {in_pkl}")

            batch_summary_rows.append({
                "setting_label": label,
                "status": "missing_input",
                "input_pkl": in_pkl,
                "output_pkl": out_pkl,
                "superwindow_minutes": np.nan,
                "super_stride_minutes": np.nan,
                "frames_per_super": np.nan,
                "final_patient_count": np.nan,
                "final_X_windows": np.nan,
                "final_y_labels": np.nan,
                "total_nan_before": np.nan,
                "total_nan_after": np.nan,
            })

            continue

        out2 = convert_pkl_to_imputed_2ch_cthw_cleaned(
            in_pkl=in_pkl,
            out_pkl=out_pkl,
            setting_label=label,
        )

        filtering_summary = out2["patient_window_filtering_summary"]
        imputation_stats = out2["imputation_stats"]

        batch_summary_rows.append({
            "setting_label": label,
            "status": "done",
            "input_pkl": in_pkl,
            "output_pkl": out_pkl,

            "superwindow_minutes": filtering_summary.get("superwindow_minutes", np.nan),
            "super_stride_minutes": filtering_summary.get("super_stride_minutes", np.nan),
            "frames_per_super": filtering_summary.get("frames_per_super", np.nan),

            "original_candidate_patient_count": filtering_summary.get(
                "original_candidate_patient_count_intersection_X_y", np.nan
            ),
            "removed_patient_count": filtering_summary.get("removed_patient_count", np.nan),
            "final_patient_count": filtering_summary.get("final_patient_count", np.nan),

            "original_candidate_X_windows": filtering_summary.get("original_candidate_X_windows", np.nan),
            "removed_X_windows": filtering_summary.get("removed_X_windows", np.nan),
            "final_X_windows": filtering_summary.get("final_X_windows", np.nan),

            "original_candidate_y_labels": filtering_summary.get("original_candidate_y_labels", np.nan),
            "removed_y_labels": filtering_summary.get("removed_y_labels", np.nan),
            "final_y_labels": filtering_summary.get("final_y_labels", np.nan),

            "total_windows_imputed": imputation_stats.get("total_windows_imputed", np.nan),
            "total_nan_before": imputation_stats.get("total_nan_before", np.nan),
            "total_nan_after": imputation_stats.get("total_nan_after", np.nan),
        })

        # -------------------------------------------------
        # Sanity check for this setting
        # -------------------------------------------------
        print("\n" + "=" * 90)
        print(f"Sanity check: {label}")
        print("=" * 90)

        if len(out2["X2_by_patient"]) > 0:
            some_pid = next(iter(out2["X2_by_patient"].keys()))
            x = out2["X2_by_patient"][some_pid]

            print("Example patient:", some_pid)
            print("X2 shape:", x.shape)
            print("NaN in value channel:", np.isnan(x[:, 0]).any())
            print("NaN in mask channel:", np.isnan(x[:, 1]).any())
        else:
            print("No valid patients found.")

        del out2
        gc.collect()

    # =====================================================
    # Save batch summary
    # =====================================================
    batch_summary_df = pd.DataFrame(batch_summary_rows)

    batch_summary_csv = os.path.join(
        OUTPUT_DIR,
        "batch_imputation_summary_loose80_2ch_CTHW.csv"
    )

    batch_summary_df.to_csv(batch_summary_csv, index=False)

    print("\n" + "=" * 100)
    print("Batch imputation finished.")
    print("=" * 100)
    print(f"Batch summary saved to: {batch_summary_csv}")
    print(batch_summary_df)